# 30. LangChain & AI SDKs — `anthropic · langchain`
**Python Libraries Lab · Phase 2 · Deep Learning & AI**

_Call large language models like Claude from your own code._

⚙️ **Runs in Colab** after the setup cell below.

The biggest language models, like Claude, are too large to run in a notebook. Instead you send them a message over the internet through an API, and get the answer back. An SDK is a small library that makes those calls easy. Anthropic's Python SDK talks to Claude directly. LangChain is a framework that wraps many providers behind one interface and adds building blocks like prompt templates and chains.

**In this notebook:** You will ask Claude a question with the Anthropic SDK, use a system prompt to make it a ShopEasy support assistant, then use LangChain to build a reusable prompt template and a chain that classifies customer reviews. You can practise the LangChain parts with a free fake model before spending any credits.

**How to use:** run each cell in order with **Shift + Enter**, or use **Runtime → Run all**. Read the short note above each cell first, then change the code and run it again: that is how you learn.

## Setup
Steps marked **needs a key** call the Claude API. Get a key at console.anthropic.com, then in Colab click the **🔑 Secrets** icon in the left sidebar, add `ANTHROPIC_API_KEY`, and turn on notebook access. Never paste a key into a cell. Every other step runs without a key.

The next cell makes sure the library is installed (Colab already has most of them; it takes a few seconds).

In [ ]:
!pip install -q anthropic langchain-anthropic langchain-core
import importlib.metadata as md
for pkg in ['anthropic', 'langchain-anthropic', 'langchain-core']:
    print(pkg, md.version(pkg))

## Step 1 · Load your API key safely
Reads the key from Colab Secrets (or an environment variable on your laptop). Prints only whether it was found.

In [ ]:
import os
API_KEY = os.environ.get("ANTHROPIC_API_KEY")
try:
    from google.colab import userdata
    API_KEY = userdata.get("ANTHROPIC_API_KEY") or API_KEY
except Exception:
    pass
if API_KEY:
    os.environ["ANTHROPIC_API_KEY"] = API_KEY
print("API key found" if API_KEY else "No API key yet: steps marked 'needs a key' will be skipped")
MODEL = "claude-opus-5-5"

## Step 2 · Ask Claude (needs a key)
One request, one reply. `max_tokens` caps the answer length.

In [ ]:
import anthropic

if API_KEY:
    client = anthropic.Anthropic()
    reply = client.messages.create(
        model=MODEL,
        max_tokens=2000,
        messages=[{"role": "user", "content": "In two sentences, what is pandas used for?"}],
    )
    print(reply.content[-1].text if reply.content else reply.stop_reason)
    print("\nTokens used:", reply.usage.input_tokens, "in,", reply.usage.output_tokens, "out")
else:
    print("Skipped: add ANTHROPIC_API_KEY in Colab Secrets to run this step.")

## Step 3 · A system prompt makes an assistant (needs a key)
The system prompt sets the role and the rules.

In [ ]:
if API_KEY:
    reply = client.messages.create(
        model=MODEL,
        max_tokens=2000,
        system="You are ShopEasy's friendly support assistant. Returns are free within 30 days. Answer in at most 3 sentences.",
        messages=[{"role": "user", "content": "My yoga mat arrived torn. What can I do?"}],
    )
    print(reply.content[-1].text)
else:
    print("Skipped: needs an API key.")

## Step 4 · LangChain prompt templates (no key needed)
A template is a prompt with blanks you fill in later.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate

prompt = ChatPromptTemplate.from_messages([
    ("system", "You classify ShopEasy reviews. Reply with exactly one word: positive, negative or mixed."),
    ("user", "Review: {review}"),
])
filled = prompt.invoke({"review": "Great lamp but delivery took two weeks."})
for m in filled.to_messages():
    print(f"{m.type:>6}: {m.content}")

## Step 5 · A chain with a practice model (no key needed)
`prompt | model | parser` is a chain. The fake model returns canned answers so you can practise for free.

In [ ]:
from langchain_core.language_models.fake_chat_models import FakeListChatModel
from langchain_core.output_parsers import StrOutputParser
reviews = [
    "Absolutely love this steel bottle, keeps water cold all day!",
    "The yoga mat smells bad and started tearing after a week. Very disappointed.",
    "Delivery was quick and the desk lamp is bright. Good value for money.",
    "Terrible customer service. My refund still hasn't arrived after 20 days.",
    "Decent headphones, but the battery life is shorter than advertised.",
    "Best purchase this year! The cotton shirts fit perfectly and feel great.",
    "The book arrived with a torn cover, but the story is wonderful.",
    "Not worth the price. The blender stopped working in two days.",
]

practice_model = FakeListChatModel(responses=["positive", "negative", "positive", "negative"])
chain = prompt | practice_model | StrOutputParser()
for r in reviews[:4]:
    print(f"{chain.invoke({'review': r}):<9} {r[:55]}")

## Step 6 · The same chain with Claude (needs a key)
Swap the practice model for the real one. Nothing else changes.

In [ ]:
if API_KEY:
    from langchain_anthropic import ChatAnthropic
    claude = ChatAnthropic(model=MODEL, max_tokens=2000)
    real_chain = prompt | claude | StrOutputParser()
    for r in reviews:
        print(f"{real_chain.invoke({'review': r}).strip():<9} {r[:55]}")
else:
    print("Skipped: needs an API key. The chain above shows the same structure.")

## ✍️ Your turn
Build a `ChatPromptTemplate` with one user message `'Translate to Hindi: {text}'`. Fill it with `text='Thank you'` and store the **filled user message content** (a string) in `answer`.

Write your code in the next cell, run it, then run the check cell.

In [ ]:
answer = None  # your code here

In [ ]:
def check():
    return answer == "Translate to Hindi: Thank you", f"Got {answer!r}"

# ▶ Run this cell to check your answer
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

### Stuck?
The next cell holds a solution. In Colab its code is hidden: double-click the cell to see it, or just run it.

In [ ]:
# @title 👀 Solution (double-click to show the code) { display-mode: "form" }
t = ChatPromptTemplate.from_messages([("user", "Translate to Hindi: {text}")])
answer = t.invoke({"text": "Thank you"}).to_messages()[0].content

# Check again with the solution
try:
    ok, msg = check()
except Exception as e:
    ok, msg = False, f"your code raised {type(e).__name__}: {e}"
print("✅ Correct! Well done." if ok else "❌ Not yet: " + msg)

## Recap
- **API key**
- **Messages**
- **System prompt**
- **Prompt template**
- **Chain**
- **Structured output**

**Next:** 31. NLTK Text Basics (`nltk`): [open in Colab](https://colab.research.google.com/github/aashaysocial27-maker/python-libraries-colab/blob/main/notebooks/31_nltk.ipynb)

## 💻 Run it on your own computer
1. Install Python 3.10+ from python.org (Windows: tick **Add python.exe to PATH**).
2. Open a terminal (Windows: *Command Prompt*; Mac: *Terminal*) and run:
```bash
mkdir python-lab && cd python-lab
python -m venv .venv
# Windows:  .venv\Scripts\activate      Mac/Linux:  source .venv/bin/activate
pip install anthropic langchain-anthropic langchain-core
pip install notebook
jupyter notebook
```
3. In Colab choose **File → Download → Download .ipynb**, then open that file in the Jupyter page that opened in your browser.

Lines that start with `!` or `%%` are notebook commands. In a plain terminal, type `!` commands without the `!`.